# 举例1：大模型分析工具的调用

In [3]:
# 1、获取大模型
# 导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.tools import StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("LLM_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("LLM_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model=os.getenv("LLM_MODEL_ID"))

# 2、获取工具的列表
tools = [MoveFileTool()]

# 3、因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数:convert_to_openai_function()
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
messages = [HumanMessage(content="将文件a移动到桌面")]

# 5、调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, # 不支持
    functions=functions,
)

print(response)

content='要在电脑上将文件 **a** 移动到桌面，具体操作取决于你使用的操作系统。下面是 **Windows** 和 **macOS** 的操作步骤：\n\n---\n\n### ✅ **Windows 系统**\n\n1. **找到文件 a**：\n   - 打开文件资源管理器（可以通过快捷键 `Win + E` 打开）。\n   - 使用搜索框或浏览路径找到文件 `a`。\n\n2. **移动文件到桌面**：\n   - 选中文件 `a`（点击一下）。\n   - 右键点击文件，选择 **“剪切”**（Cut）。\n   - 进入桌面，右键点击空白处，选择 **“粘贴”**（Paste）。\n   - 或者你也可以用快捷键：\n     - `Ctrl + X`（剪切）\n     - `Ctrl + V`（粘贴）在桌面上操作。\n\n---\n\n### ✅ **macOS 系统**\n\n1. **找到文件 a**：\n   - 打开 Finder（在 Dock 上）。\n   - 找到文件 `a`，可以使用搜索功能（命令 + 空格）。\n\n2. **移动文件到桌面**：\n   - 选中文件 `a`。\n   - 拖动文件到桌面（鼠标拖拽或使用快捷键 `Command + C` 复制，`Command + V` 粘贴）。\n   - 或者右键点击文件，选择 **“复制”**（Copy），然后在桌面右键选择 **“粘贴”**。\n\n---\n\n### ⚠️ 注意事项：\n- 如果你不知道文件 `a` 在哪里，可以通过“搜索”功能查找。\n- 如果文件在某个文件夹中，可以直接拖到桌面，或者右键选择“移动到桌面”。\n- 如果你使用的是 **Linux** 系统，也可以用命令行操作：\n\n---\n\n### ✅ **Linux 系统（如 Ubuntu）**\n\n```bash\nmv a ~/Desktop\n```\n\n- 假设你当前在文件所在目录。\n- `mv` 是移动命令。\n- `a` 是要移动的文件名。\n- `~/Desktop` 是你的桌面目录。\n\n---\n\n如果你告诉我你使用的是哪种操作系统，我可以给你更具体的帮助！' additional_kwargs={'refusal': None} response_me

作为对比：


In [ ]:
# 获取消息列表
messages = [HumanMessage(content="查询一下明天北京的天气")]

# 调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

print(response)

content='好的，以下是明天（2024年4月22日）北京的天气情况（具体数据可能会有微调，请以实时天气预报为准）：\n\n**北京明天的天气概况：**\n\n- **天气状况**：晴转多云\n- **气温**：白天最高气温约 **18°C**，夜间最低气温约 **4°C**\n- **风力**：东北风 2-3 级\n- **空气质量**：良，适宜户外活动\n- **降水概率**：较低，大约 **5%**\n\n---\n\n如果你想知道更详细的信息（例如每小时的温度变化、湿度、紫外线强度等），可以使用天气预报App（如墨迹天气、彩云天气、AccuWeather）或者访问中国天气网（http://www.weather.com.cn）查询。也可以让我帮你用天气接口获取实时数据。需要的话请告诉我。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 186, 'prompt_tokens': 18, 'total_tokens': 204, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': '', 'id': '019c27d52e8d779c60b2f530699ed021', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c27d5-3972-7312-a93f-1483e41e7ab1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 186, 'total_to

通过上面两个测试发现，得到的AIMessage的核心属性如下：

1、如果分析出需要调用对应的工具：

content：信息为空。因为大模型要调用工具，所以就不会直接返回信息给用户

additional_kwargs：包含function_call字段，指明具体函数调用的参数和函数名。比如：

additional_kwargs={'function_call': {'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}', 'name': 'move_file'}, 'refusal': None}

2、如果分析出不需要调用对应的工具：

content：信息不为空。

additional_kwargs：不包含function_call字段








# 举例2：如何调用具体大模型分析出来的工具

说明：

1、大模型与Agent的核心区别：是否涉及到工具的调用

2、针对于大模型：仅能分析出要调用的工具，但是此工具（或函数）不能真正的执行

   针对于Agent：除了分析出要调用的工具之外，还可以执行具体的工具（或函数）

In [24]:
# 1、获取大模型
# 导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("LLM_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("LLM_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model=os.getenv("LLM_MODEL_ID"))

# 2、获取工具的列表
tools = [MoveFileTool()]

# 3、使用bind_tools方法将工具绑定到模型（推荐方式）
# chat_model_with_tools = chat_model.bind_tools(tools)

# 4、获取消息列表（添加更明确的指令）
messages = [HumanMessage(content="请使用可用的工具将当前目录下的文件a.txt移动到E:\\Desktop\\目录下")]

# 5、调用带工具的大模型
# response = chat_model_with_tools.invoke(messages)
functions = [convert_to_openai_function(t) for t in tools]
response = chat_model.invoke(input=messages, functions=functions)

print(response)

content='要在当前目录下将文件 `a.txt` 移动到 `E:\\Desktop\\` 目录下，你可以使用以下几种方法：\n\n---\n\n### ✅ 方法一：使用命令提示符（CMD）\n\n1. 打开 **命令提示符**（可按下 `Win + R`，输入 `cmd`，然后回车）。\n2. 执行下面的命令：\n\n```cmd\nmove a.txt E:\\Desktop\\\n```\n\n这会将当前目录下的 `a.txt` 文件移动到 `E:\\Desktop` 目录中。\n\n---\n\n### ✅ 方法二：使用 PowerShell\n\n1. 打开 **PowerShell**（可按 `Win + R`，输入 `powershell`，然后回车）。\n2. 执行下面的命令：\n\n```powershell\nMove-Item -Path .\\a.txt -Destination "E:\\Desktop"\n```\n\n同样会将当前目录下的 `a.txt` 文件移动到 `E:\\Desktop`。\n\n---\n\n### ✅ 方法三：使用图形界面（Windows 资源管理器）\n\n1. 打开 **文件资源管理器**。\n2. 导航到当前目录（即 `a.txt` 所在的文件夹）。\n3. 找到文件 `a.txt`，右键点击它 → 选择 **剪切**。\n4. 导航到 `E:\\Desktop` 目录。\n5. 右键点击空白处 → 选择 **粘贴**。\n\n---\n\n### ⚠️ 注意事项：\n\n- 确保 `E:\\Desktop` 目录存在，否则移动会失败。\n- 如果 `a.txt` 是只读文件，可能需要取消只读属性才能移动。\n- 如果有权限限制，请以管理员身份运行命令提示符或 PowerShell。\n- 如果你是在某个特定路径下操作，比如 `C:\\Users\\Name\\Documents`，请将 `.\\a.txt` 替换为实际文件路径。\n\n---\n\n如你有特定的背景或环境需求（如 Python 脚本），也可以告诉我，我可以提供相应的代码实现。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'comple

步骤1：分析下要调用哪个工具或函数

In [25]:
import json

# 检查是否有工具调用（新版LangChain使用tool_calls）
if response.tool_calls:
    tool_call = response.tool_calls[0]
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    print(f"调用工具：{tool_name}")
    print(f"参数：{tool_args}")
    
# 兼容旧版本的function_call方式
elif "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具：{tool_name}")
    print(f"参数：{tool_args}")
    
else:
    print(f"模型回复：{response.content}")
    print("\n注意：模型没有调用工具，可能的原因：")
    print("1. 模型不支持function calling")
    print("2. 提示词不够明确")
    print("3. 模型认为不需要使用工具")

模型回复：要在当前目录下将文件 `a.txt` 移动到 `E:\Desktop\` 目录下，你可以使用以下几种方法：

---

### ✅ 方法一：使用命令提示符（CMD）

1. 打开 **命令提示符**（可按下 `Win + R`，输入 `cmd`，然后回车）。
2. 执行下面的命令：

```cmd
move a.txt E:\Desktop\
```

这会将当前目录下的 `a.txt` 文件移动到 `E:\Desktop` 目录中。

---

### ✅ 方法二：使用 PowerShell

1. 打开 **PowerShell**（可按 `Win + R`，输入 `powershell`，然后回车）。
2. 执行下面的命令：

```powershell
Move-Item -Path .\a.txt -Destination "E:\Desktop"
```

同样会将当前目录下的 `a.txt` 文件移动到 `E:\Desktop`。

---

### ✅ 方法三：使用图形界面（Windows 资源管理器）

1. 打开 **文件资源管理器**。
2. 导航到当前目录（即 `a.txt` 所在的文件夹）。
3. 找到文件 `a.txt`，右键点击它 → 选择 **剪切**。
4. 导航到 `E:\Desktop` 目录。
5. 右键点击空白处 → 选择 **粘贴**。

---

### ⚠️ 注意事项：

- 确保 `E:\Desktop` 目录存在，否则移动会失败。
- 如果 `a.txt` 是只读文件，可能需要取消只读属性才能移动。
- 如果有权限限制，请以管理员身份运行命令提示符或 PowerShell。
- 如果你是在某个特定路径下操作，比如 `C:\Users\Name\Documents`，请将 `.\a.txt` 替换为实际文件路径。

---

如你有特定的背景或环境需求（如 Python 脚本），也可以告诉我，我可以提供相应的代码实现。

注意：模型没有调用工具，可能的原因：
1. 模型不支持function calling
2. 提示词不够明确
3. 模型认为不需要使用工具


步骤2：调用对应的工具

In [26]:
# 检查并执行工具调用
if response.tool_calls:
    tool_call = response.tool_calls[0]
    if "move_file" in tool_call["name"]:
        tool = MoveFileTool()
        result = tool.run(tool_call["args"])
        print("工具执行的结果:", result)
        
elif "function_call" in response.additional_kwargs:
    # 旧版方式
    if "move_file" in response.additional_kwargs["function_call"]["name"]:
        tool = MoveFileTool()
        tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
        result = tool.run(tool_args)
        print("工具执行的结果:", result)
else:
    print("模型没有返回工具调用，无法执行")

模型没有返回工具调用，无法执行
